In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
import torch

import os
from dotenv import load_dotenv

load_dotenv()

login(os.getenv("HF_TOKEN"))

In [ ]:
base_model = "sinister007/llama-3.2-1b-mcq-gen"


tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map = "cuda"
)


In [ ]:
prompt = """Functions In C++
Objective:
- To understand the concept of functions in C++, including their definition, purpose, and
usage in breaking down a program into manageable components.
4.1 Introduction to Functions
Functions are an essential concept in programming, allowing you to break down complex
problems into smaller, more manageable parts. In C++, a function is a block of code that
performs a specific task, and it can be called upon whenever that task needs to be executed.
Functions help in organizing code, making it reusable, and improving readability.
4.2 Function Definition and Declaration
A function in C++ consists of two main parts: the function declaration and the function
definition.
- Function Declaration: Also known as a function prototype, it specifies the function’s
name, return type, and parameters (if any). The declaration informs the compiler about
the function, ensuring that it can be called from other parts of the program.
- Function Definition: This includes the actual code block that defines what the function
does. It contains the statements that will be executed when the function is called. The
definition also specifies the return type, parameters, and the function body.
4.3 Return Type and Parameters
Functions can return a value after execution, which is specified by the return type. For instance, a
function might return an integer, a floating-point number, or a character. If no value is returned,
the return type is `void`.
Functions can also accept input values, known as parameters or arguments. Parameters allow
functions to process data and return results based on different inputs. Parameters are specified in
the function’s declaration and definition, and they enable the same function to be used in various
contexts with different data.
Functions In C++
3
4.4 Calling a Function
To use a function, you "call" it from another part of the program. When a function is called, the
program's control is transferred to the function, and the function's code is executed. After the
function finishes executing, control is returned to the point in the program where the function
was called.
Calling a function involves passing any required arguments to the function. The function
processes these arguments and returns a result, if applicable. The use of functions in a program
allows for better structure and helps avoid redundancy by enabling code reuse.
4.5 Function Overloading
C++ allows multiple functions to have the same name, as long as they have different parameter
lists. This is known as function overloading. Overloading enables you to create functions that
perform similar tasks but work with different types or numbers of parameters.
Function overloading improves the flexibility of your code, allowing you to handle various types
of input without needing to create entirely separate functions with different names.
4.6 Scope and Lifetime of Variables
In C++, the scope of a variable refers to the part of the program where the variable can be
accessed. Variables declared within a function are local to that function, meaning they cannot be
accessed from outside the function. The lifetime of these local variables is limited to the
function's execution; they are created when the function is called and destroyed when the
function ends.
Understanding the scope and lifetime of variables is crucial for writing functions that work
correctly and avoid unintended side effects in the program.
4.7 Recursion
Recursion is a concept where a function calls itself to solve a smaller instance of the same
problem. Recursive functions can be powerful, but they must be carefully designed to ensure
they have a base case to terminate the recursion and avoid infinite loops.
Recursion is particularly useful for tasks that can be broken down into simpler, identical tasks,
such as mathematical computations or navigating data structures like trees.
Functions In C++
4.8 Best Practices for Using Functions
Using functions effectively involves following best practices, including:
- Modularity: Divide your program into functions that each perform a specific task. This
modular approach makes your program easier to understand, test, and maintain.
- Function Naming: Choose descriptive names for your functions that reflect their
purpose. This makes your code more readable and easier to debug.
- Minimize Side Effects: Avoid modifying global variables within functions to prevent
unintended side effects. Instead, use function parameters and return values to pass data in
and out of functions.
Conclusion
Functions are a critical part of C++ programming, enabling you to write cleaner, more organized,
and reusable code. By mastering functions, you can effectively manage the complexity of your
programs, making them more efficient and easier to maintain. Understanding how to define,
declare, and use functions is fundamental to becoming proficient in C++.
"""

In [ ]:
def create_test_prompt():
    messages = [
        {
            "role": "system",
            "content": f"""You are a Computer Science teacher. Generate exactly 20 hard multiple choice questions from the context below.
Each MCQ must follow this format:
Question: ...
A. ...
B. ...
C. ...
D. ...
Answer: X
Ensure there is no repetition. Questions must be meaningful and test deep understanding.""",
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [ ]:
pp = create_test_prompt()
inputs = tokenizer(pp, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_length=2000,
    do_sample=True,
    temperature=0.6,
)

In [ ]:
generated_tokens = outputs[0]

# Get the number of tokens in the input (prompt)
prompt_length = inputs["input_ids"].shape[1]

# Slice the generated tokens to remove the prompt part
generated_tokens_only = generated_tokens[prompt_length:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_tokens_only, skip_special_tokens=True)
print(output_text)